[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_29/2_sql.ipynb)

# Day 29: SQL full round (SQL and pandas)

**Full exam round (45 minutes): an analytical SQL round with pandas follow-ups.**

- Treat this as the real thing: 5 tasks, about 9 minutes each, one timer. For each task write the **SQL first**
  (the main ask), then the **pandas** version (the usual follow-up, and what a take-home expects).
- Talk while you work: state the grain (one row = ?), the definition, and one edge case before you type.
- The check cell prints PASS when SQL and pandas agree with each other and with the expected answer.
- No hints until the 45 minutes are over. Afterwards, read every solution: the pandas idioms are the part to memorise.

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (45 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Monthly revenue and growth | hard | 8 |
| Q2 | Best movies per genre | hard | 9 |
| Q3 | Two week conversion by channel and month | hard | 9 |
| Q4 | Sessions from an event log | hard | 10 |
| Q5 | Funnel table by platform | hard | 9 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Tables

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real table: Online Retail** (UCI, a UK online gift shop, 2010-12-01 to 2011-12-09, CC BY 4.0):

| Table | One row is | Columns |
|---|---|---|
| `retail` | one product line of an invoice (541,909) | `invoice_no` (text; a 'C' prefix = cancellation), `stock_code`, `description`, `quantity` (negative for cancellations), `invoice_date` ('YYYY-MM-DD HH:MM:SS'), `unit_price` (GBP), `customer_id` (NULL for about 25% of rows), `country` |

**Real tables: MovieLens small** (GroupLens; 610 users rated 9,742 movies, 1996 to 2018):

| Table | One row is | Columns |
|---|---|---|
| `ml_ratings` | one rating (100,836) | `user_id`, `movie_id`, `rating` (0.5 to 5.0 in steps of 0.5), `rated_at` ('YYYY-MM-DD HH:MM:SS', UTC) |
| `ml_movies` | one movie (9,742) | `movie_id`, `title`, `year` (from the title, can be NULL), `genres` (pipe separated, e.g. 'Action|Comedy') |
| `ml_movie_genres` | one (movie, genre) pair | `movie_id`, `genre` (the `genres` string split into rows) |

**pandas:** the setup cell also loads `retail`, `ml_ratings`, `ml_movies`, `ml_movie_genres`, `app_users`,
`app_events` and `purchases` as DataFrames (same names and columns as the SQL tables). `same(sql, df, ...)` checks a
task: SQL and pandas must agree, and the SQL result must have the expected size and first row.

The setup cell prints every table with its columns. Call `tables()` again any time.

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: UCI Online Retail (a UK gift shop, 2010-12 to 2011-12, CC BY 4.0).
import io, zipfile
_p = os.path.join(DATA, "online_retail.csv.gz")
if not os.path.exists(_p):                       # first run in Colab: download the UCI zip and convert it once
    with zipfile.ZipFile(data_path("online_retail.zip", "https://archive.ics.uci.edu/static/public/352/online+retail.zip")) as zf:
        pd.read_excel(io.BytesIO(zf.read("Online Retail.xlsx"))).to_csv(_p, index=False, compression="gzip")
_o = pd.read_csv(_p, dtype={"InvoiceNo": str, "StockCode": str})
_o.columns = ["invoice_no", "stock_code", "description", "quantity", "invoice_date", "unit_price", "customer_id", "country"]
_o["invoice_date"] = pd.to_datetime(_o["invoice_date"]).dt.strftime("%Y-%m-%d %H:%M:%S")
_o["customer_id"] = _o["customer_id"].astype("Int64")
_o.to_sql("retail", db, index=False)

# Real data: MovieLens small (GroupLens, research and education use). Downloads the zip if the CSVs are missing.
import zipfile
if not os.path.exists(os.path.join(DATA, "movielens_ratings.csv")):
    _z = data_path("ml-latest-small.zip", "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip")
    with zipfile.ZipFile(_z) as zf:
        for _n in ("ratings", "movies"):
            with open(os.path.join(DATA, f"movielens_{_n}.csv"), "wb") as fh:
                fh.write(zf.read(f"ml-latest-small/{_n}.csv"))
_r = pd.read_csv(data_path("movielens_ratings.csv"))
_r["rated_at"] = pd.to_datetime(_r["timestamp"], unit="s").dt.strftime("%Y-%m-%d %H:%M:%S")
_r = _r.rename(columns={"userId": "user_id", "movieId": "movie_id"})[["user_id", "movie_id", "rating", "rated_at"]]
_r.to_sql("ml_ratings", db, index=False)
_m = pd.read_csv(data_path("movielens_movies.csv")).rename(columns={"movieId": "movie_id"})
_m["year"] = pd.to_numeric(_m["title"].str.extract(r"\((\d{4})\)\s*$")[0], errors="coerce").astype("Int64")
_m[["movie_id", "title", "year", "genres"]].to_sql("ml_movies", db, index=False)
_g = _m.assign(genre=_m["genres"].str.split("|")).explode("genre")[["movie_id", "genre"]]
_g[_g["genre"] != "(no genres listed)"].to_sql("ml_movie_genres", db, index=False)

# pandas copies of the tables (same names as the SQL tables) and the answer checker.
import numpy as np
retail = pd.read_sql("SELECT * FROM retail", db)
ml_ratings = pd.read_sql("SELECT * FROM ml_ratings", db)
ml_movies = pd.read_sql("SELECT * FROM ml_movies", db)
ml_movie_genres = pd.read_sql("SELECT * FROM ml_movie_genres", db)
app_users = pd.read_sql("SELECT * FROM app_users", db)
app_events = pd.read_sql("SELECT * FROM app_events", db)
purchases = pd.read_sql("SELECT * FROM purchases", db)

def _norm(t):
    t = pd.DataFrame(t).reset_index(drop=True).copy()
    t.columns = range(t.shape[1])
    for c in t.columns:
        conv = pd.to_numeric(t[c], errors="coerce")
        if conv.notna().sum() == t[c].notna().sum():          # a numeric column (NULLs allowed)
            t[c] = conv.astype(float)
        else:
            t[c] = t[c].astype(str)
    return t

def _close(a, b):
    if isinstance(a, float) or isinstance(b, float):
        try:
            a, b = float(a), float(b)
        except (TypeError, ValueError):
            return False
        return (np.isnan(a) and np.isnan(b)) or abs(a - b) <= 0.06
    return str(a) == str(b)

def same(sql, df, rows=None, first=None):
    """Checks one task: your SQL result and your pandas result must hold the same rows (column names and row order
    are ignored, numbers may differ by rounding), and match the expected row count and first row of the SQL result."""
    if not str(sql).strip() or df is None:
        print("Not done yet: write the SQL string and the pandas function, then run this cell again.")
        return
    a, b = _norm(run(sql)), _norm(df)
    if a.shape != b.shape:
        print(f"FAIL SQL gives {a.shape[0]} rows x {a.shape[1]} columns, pandas gives {b.shape[0]} x {b.shape[1]}")
    else:
        sa = a.sort_values(list(a.columns)).reset_index(drop=True)
        sb = b.sort_values(list(b.columns)).reset_index(drop=True)
        bad = [(i, j) for i in range(len(sa)) for j in sa.columns if not _close(sa.iat[i, j], sb.iat[i, j])]
        print("PASS SQL and pandas agree" if not bad else f"FAIL SQL and pandas differ, e.g. row {bad[0][0]} column {bad[0][1]}")
    if rows is not None:
        print(("PASS" if a.shape[0] == rows else "FAIL") + f" expected {rows} rows, SQL gives {a.shape[0]}")
    if first is not None:
        got = [v.item() if hasattr(v, "item") else v for v in a.iloc[0]] if len(a) else []
        got = [int(v) if isinstance(v, float) and v.is_integer() else v for v in got]
        ok = len(got) == len(first) and all(_close(float("nan") if f is None else f, g) for f, g in zip(first, got))
        print(("PASS" if ok else "FAIL") + f" first row of the SQL result: {got}")

def tables():
    """Prints every table with its row count and columns."""
    for (t,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
        cols = [r[1] for r in db.execute(f'PRAGMA table_info("{t}")')]
        n = db.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        print(f"{t} ({n:,} rows): {', '.join(cols)}")

tables()

---
### Q1. Monthly revenue and growth

*hard, about 8 min*

UCI Online Retail (real). Net revenue = `SUM(quantity * unit_price)` over ALL rows (cancellations have
negative quantities, so they reduce revenue). Use full months only: `invoice_date < '2011-12-01'`. Return `month`
('YYYY-MM'), `revenue` (rounded to whole numbers) and `mom_pct`: month over month change in % (1 decimal; NULL for the
first month). Order by month.

pandas: return the same three columns.

In [ ]:
q1_sql = """

"""

def q1_pandas():
    # your pandas code: return a DataFrame with the same columns
    return None

print(run(q1_sql) if q1_sql.strip() else "")

# tests
same(q1_sql, q1_pandas(), rows=12, first=("2010-12", 748957.0, None))

<details><summary><b>Hint 1</b></summary>

Signal: 'compared with the previous month'. Pattern: aggregate by month, then LAG (SQL) / `pct_change` or `shift` (pandas).

</details>

<details><summary><b>Hint 2</b></summary>

SQL: CTE with SUM per `substr(invoice_date, 1, 7)`, then `revenue / LAG(revenue) OVER (ORDER BY month) - 1`.
pandas: filter, `assign(month=..., rev=...)`, `groupby('month', as_index=False)['rev'].sum()`, then `rev.pct_change() * 100`.

</details>

<details><summary><b>Solution</b></summary>

```python
q1_sql = """
WITH m AS (
  SELECT substr(invoice_date, 1, 7) AS month, SUM(quantity * unit_price) AS revenue
  FROM retail
  WHERE invoice_date < '2011-12-01'
  GROUP BY substr(invoice_date, 1, 7)
)
SELECT month, ROUND(revenue, 0) AS revenue,
  ROUND(100.0 * (revenue / LAG(revenue) OVER (ORDER BY month) - 1), 1) AS mom_pct
FROM m
ORDER BY month
"""

def q1_pandas():
    d = retail[retail["invoice_date"] < "2011-12-01"]
    d = d.assign(month=d["invoice_date"].str[:7], rev=d["quantity"] * d["unit_price"])
    m = d.groupby("month", as_index=False)["rev"].sum().sort_values("month")
    m["mom_pct"] = (m["rev"].pct_change() * 100).round(1)
    m["revenue"] = m["rev"].round(0)
    return m[["month", "revenue", "mom_pct"]]
```

**Why:** LAG in SQL and `pct_change` (or `rev / rev.shift(1) - 1`) in pandas both need the rows sorted by month; groupby sorts its keys, and the explicit sort_values makes that visible. November 2011 is the peak (1,461,756, +36.5%) before the gift season; the partial December is excluded on purpose, otherwise it shows a fake crash.

**Complexity:** One pass to aggregate; the monthly table has 12 rows.

**Common mistakes:** Leaving December 2011 (9 days) in. `pct_change` on an unsorted frame. pandas mean of the column when the task says sum. Forgetting `as_index=False` and then losing the month column.

**Learn more:** [sql-pandas-equivalents](https://mahsa-agz.github.io/ds-handbook/#sql-pandas-equivalents), [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead)

</details>

---
### Q2. Best movies per genre

*hard, about 9 min*

MovieLens (real). For the genres Comedy, Drama and Sci-Fi (`ml_movie_genres`), consider movies with at least
100 ratings. Return the **top 3 per genre** by average rating (ties: more ratings first): `genre`, `rn` (1 to 3),
`title`, `n` (ratings), `avg_rating` (2 decimals). Order by genre, rn.

pandas: same columns.

In [ ]:
q2_sql = """

"""

def q2_pandas():
    # your pandas code: return a DataFrame with the same columns
    return None

print(run(q2_sql) if q2_sql.strip() else "")

# tests
same(q2_sql, q2_pandas(), rows=9, first=("Comedy", 1, "Princess Bride, The (1987)", 142, 4.23))

<details><summary><b>Hint 1</b></summary>

Signal: 'top 3 per genre'. Pattern: top-N per group: ROW_NUMBER (SQL); sort + `groupby().cumcount()` or `groupby().head(3)` (pandas).

</details>

<details><summary><b>Hint 2</b></summary>

SQL: stats CTE (GROUP BY genre, movie HAVING COUNT(*) >= 100), then `ROW_NUMBER() OVER (PARTITION BY genre ORDER BY avg_rating DESC, n DESC)`.
pandas: merge ratings with the filtered genres, `groupby(['genre', 'movie_id']).agg(n=('rating', 'size'), avg_rating=('rating', 'mean'))`, filter n, merge titles, sort, `cumcount() + 1`.

</details>

<details><summary><b>Solution</b></summary>

```python
q2_sql = """
WITH stats AS (
  SELECT g.genre, m.title, COUNT(*) AS n, AVG(r.rating) AS avg_rating
  FROM ml_ratings r
  JOIN ml_movie_genres g ON g.movie_id = r.movie_id
  JOIN ml_movies m ON m.movie_id = r.movie_id
  WHERE g.genre IN ('Comedy', 'Drama', 'Sci-Fi')
  GROUP BY g.genre, r.movie_id, m.title
  HAVING COUNT(*) >= 100
), ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY genre ORDER BY avg_rating DESC, n DESC) AS rn
  FROM stats
)
SELECT genre, rn, title, n, ROUND(avg_rating, 2) AS avg_rating
FROM ranked
WHERE rn <= 3
ORDER BY genre, rn
"""

def q2_pandas():
    g = ml_movie_genres[ml_movie_genres["genre"].isin(["Comedy", "Drama", "Sci-Fi"])]
    s = (ml_ratings.merge(g, on="movie_id")
         .groupby(["genre", "movie_id"])
         .agg(n=("rating", "size"), avg_rating=("rating", "mean"))
         .reset_index())
    s = s[s["n"] >= 100].merge(ml_movies[["movie_id", "title"]], on="movie_id")
    s = s.sort_values(["genre", "avg_rating", "n"], ascending=[True, False, False])
    s["rn"] = s.groupby("genre").cumcount() + 1
    out = s[s["rn"] <= 3][["genre", "rn", "title", "n", "avg_rating"]]
    return out.assign(avg_rating=out["avg_rating"].round(2))
```

**Why:** HAVING and the pandas boolean filter both run AFTER the aggregation (a WHERE on n would not exist yet). `sort_values` + `groupby().cumcount()` is the pandas ROW_NUMBER; `groupby('genre').head(3)` works too once sorted. Shawshank Redemption leads Drama with 4.43 from 317 ratings; Pulp Fiction counts as a Comedy too (multi-genre movies appear in each of their genres).

**Complexity:** A join of 100k ratings to genres, one groupby and one sort.

**Common mistakes:** `rank(method='first')` without sorting by the tiebreaker. Filtering n before grouping. Using `nlargest` per group with apply (slow and drops the tiebreaker).

**Learn more:** [sql-pandas-equivalents](https://mahsa-agz.github.io/ds-handbook/#sql-pandas-equivalents), [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n)

</details>

---
### Q3. Two week conversion by channel and month

*hard, about 9 min*

Shopping app (made up). A user **converts** if they have a `completed` purchase within 14 days of signup
(`order_time < signup_date + 14 days`). Return `signup_month` ('YYYY-MM'), `channel`, `users`, `conv_14d_pct` (1
decimal), ordered by month, then conversion descending. Users without purchases must count.

pandas: same columns.

In [ ]:
q3_sql = """

"""

def q3_pandas():
    # your pandas code: return a DataFrame with the same columns
    return None

print(run(q3_sql) if q3_sql.strip() else "")

# tests
same(q3_sql, q3_pandas(), rows=8, first=("2024-01", "paid_social", 227, 21.6))

<details><summary><b>Hint 1</b></summary>

Signal: a rate where non-buyers must stay in the denominator. Pattern: LEFT JOIN with the filter in ON + MAX(CASE) flag per user (SQL); `isin` flag + groupby mean (pandas).

</details>

<details><summary><b>Hint 2</b></summary>

SQL: per user `MAX(CASE WHEN p.order_time < date(u.signup_date, '+14 days') THEN 1 ELSE 0 END)` from `app_users LEFT JOIN purchases ... AND status = 'completed'`; then AVG per month and channel.
pandas: merge completed purchases with signup dates, keep those inside 14 days, flag users with `isin`, then `groupby(['signup_month', 'channel']).agg(users=..., conv=('flag', 'mean'))`.

</details>

<details><summary><b>Solution</b></summary>

```python
q3_sql = """
WITH conv AS (
  SELECT u.user_id, u.channel, substr(u.signup_date, 1, 7) AS signup_month,
    MAX(CASE WHEN p.order_time < date(u.signup_date, '+14 days') THEN 1 ELSE 0 END) AS bought_14d
  FROM app_users u
  LEFT JOIN purchases p ON p.user_id = u.user_id AND p.status = 'completed'
  GROUP BY u.user_id, u.channel, u.signup_date
)
SELECT signup_month, channel, COUNT(*) AS users,
  ROUND(100.0 * AVG(bought_14d), 1) AS conv_14d_pct
FROM conv
GROUP BY signup_month, channel
ORDER BY signup_month, conv_14d_pct DESC
"""

def q3_pandas():
    p = purchases[purchases["status"] == "completed"].merge(app_users[["user_id", "signup_date"]], on="user_id")
    inside = pd.to_datetime(p["order_time"]) < pd.to_datetime(p["signup_date"]) + pd.Timedelta(days=14)
    buyers = p.loc[inside, "user_id"].unique()
    u = app_users.assign(signup_month=app_users["signup_date"].str[:7],
                         bought_14d=app_users["user_id"].isin(buyers).astype(int))
    o = (u.groupby(["signup_month", "channel"])
          .agg(users=("user_id", "size"), conv_14d_pct=("bought_14d", "mean"))
          .reset_index())
    o["conv_14d_pct"] = (o["conv_14d_pct"] * 100).round(1)
    return o.sort_values(["signup_month", "conv_14d_pct"], ascending=[True, False])
```

**Why:** Both versions build ONE 0/1 flag per user first and only then average, so users with several orders count once and users with none count as 0. Paid social converts best in January (21.6%) but worst in February (15.7%): a channel ranking that flips month to month needs a significance check before anyone acts on it.

**Complexity:** One join and two aggregations in both versions.

**Common mistakes:** Inner merge in pandas (non-buyers vanish). `status = 'completed'` in WHERE after the LEFT JOIN. Comparing a date string with a timestamp string in pandas without `to_datetime`.

**Learn more:** [sql-pandas-equivalents](https://mahsa-agz.github.io/ds-handbook/#sql-pandas-equivalents), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios)

</details>

---
### Q4. Sessions from an event log

*hard, about 10 min*

Shopping app (made up), `app_events`. A new session starts when a user's event comes more than 30 minutes
after their previous event (or is their first event). Per `platform` (from `app_users`) return `sessions`,
`events_per_session` (2 decimals) and `single_event_pct` (% of sessions with only one event, 1 decimal), ordered by
platform. Break timestamp ties with `event_id`.

pandas: same columns. (Today's algorithms notebook asks the same in plain Python.)

In [ ]:
q4_sql = """

"""

def q4_pandas():
    # your pandas code: return a DataFrame with the same columns
    return None

print(run(q4_sql) if q4_sql.strip() else "")

# tests
same(q4_sql, q4_pandas(), rows=3, first=("android", 5065, 1.71, 60.2))

<details><summary><b>Hint 1</b></summary>

Signal: '30 minutes after the previous event'. Pattern: sessionization: LAG + flag + running SUM (SQL); `groupby().diff()` + flag + `groupby().cumsum()` (pandas).

</details>

<details><summary><b>Hint 2</b></summary>

SQL: LAG(event_time) per user; new_s = 1 if NULL or gap > 30 min; `SUM(new_s) OVER (PARTITION BY user_id ORDER BY event_time, event_id ROWS UNBOUNDED PRECEDING)` = session id; count events per session.
pandas: sort by user, time, id; `gap = t.groupby(user).diff()`; `flag = gap.isna() | (gap > 30 min)`; `sid = flag.astype(int).groupby(user).cumsum()`.

</details>

<details><summary><b>Solution</b></summary>

```python
q4_sql = """
WITH e AS (
  SELECT e.user_id, u.platform, e.event_id, e.event_time,
    LAG(e.event_time) OVER (PARTITION BY e.user_id ORDER BY e.event_time, e.event_id) AS prev_time
  FROM app_events e
  JOIN app_users u ON u.user_id = e.user_id
), f AS (
  SELECT *,
    CASE WHEN prev_time IS NULL
           OR (julianday(event_time) - julianday(prev_time)) * 1440 > 30 THEN 1 ELSE 0 END AS new_s
  FROM e
), s AS (
  SELECT user_id, platform,
    SUM(new_s) OVER (PARTITION BY user_id ORDER BY event_time, event_id ROWS UNBOUNDED PRECEDING) AS sid
  FROM f
), per AS (
  SELECT platform, user_id, sid, COUNT(*) AS events FROM s GROUP BY platform, user_id, sid
)
SELECT platform, COUNT(*) AS sessions,
  ROUND(AVG(events), 2) AS events_per_session,
  ROUND(100.0 * AVG(CASE WHEN events = 1 THEN 1 ELSE 0 END), 1) AS single_event_pct
FROM per
GROUP BY platform
ORDER BY platform
"""

def q4_pandas():
    e = (app_events.merge(app_users[["user_id", "platform"]], on="user_id")
         .sort_values(["user_id", "event_time", "event_id"]))
    t = pd.to_datetime(e["event_time"])
    gap = t.groupby(e["user_id"]).diff()
    new_s = gap.isna() | (gap > pd.Timedelta(minutes=30))
    e["sid"] = new_s.astype(int).groupby(e["user_id"]).cumsum()
    per = e.groupby(["platform", "user_id", "sid"]).size().rename("events").reset_index()
    out = per.groupby("platform").agg(sessions=("events", "size"),
                                      events_per_session=("events", "mean"),
                                      single_event_pct=("events", lambda x: (x == 1).mean() * 100)).reset_index()
    return out.round({"events_per_session": 2, "single_event_pct": 1})
```

**Why:** The flag-and-cumsum idea is identical in both languages: a boolean 'starts a new session' turned into a running count per user. In pandas, `groupby(...).diff()` must be used (a plain `diff()` would compare the first event of a user with the last event of the previous user). Android has 5,065 sessions with 1.71 events each; about 60% of sessions on every platform are a single event, a typical bounce signal.

**Complexity:** One sort by user and time: O(n log n); everything else is linear.

**Common mistakes:** Plain `diff()` across users. Sorting by time only (users interleave). Gap measured in the wrong unit (`gap > 30` compares a Timedelta with an int: error). Forgetting the tie breaker.

**Learn more:** [sql-pandas-equivalents](https://mahsa-agz.github.io/ds-handbook/#sql-pandas-equivalents), [sql-gaps-islands](https://mahsa-agz.github.io/ds-handbook/#sql-gaps-islands)

</details>

---
### Q5. Funnel table by platform

*hard, about 9 min*

Shopping app (made up). Per `platform` return `users` (all users of the platform), and the number of users
who EVER had each event: `viewed`, `carted`, `checked_out`, `purchased`, plus `view_to_buy_pct` = purchased / viewed
(1 decimal). Order matters NOT here (the strict ordered funnel was day 21). Order by platform.

pandas: same columns, using a pivot table.

In [ ]:
q5_sql = """

"""

def q5_pandas():
    # your pandas code: return a DataFrame with the same columns
    return None

print(run(q5_sql) if q5_sql.strip() else "")

# tests
same(q5_sql, q5_pandas(), rows=3, first=("android", 695, 642, 522, 421, 205, 31.9))

<details><summary><b>Hint 1</b></summary>

Signal: counts per category as columns. Pattern: pivot: COUNT(DISTINCT CASE ...) (SQL); `drop_duplicates` + `pivot_table(aggfunc='count')` or `pd.crosstab` (pandas).

</details>

<details><summary><b>Hint 2</b></summary>

SQL: `app_users LEFT JOIN app_events`, one `COUNT(DISTINCT CASE WHEN e.event_name = 'view_item' THEN u.user_id END)` per step.
pandas: `app_events.drop_duplicates(['user_id', 'event_name'])`, merge platform, `pivot_table(index='platform', columns='event_name', values='user_id', aggfunc='count', fill_value=0)`, join the user counts, reorder the columns.

</details>

<details><summary><b>Solution</b></summary>

```python
q5_sql = """
SELECT u.platform,
  COUNT(DISTINCT u.user_id) AS users,
  COUNT(DISTINCT CASE WHEN e.event_name = 'view_item' THEN u.user_id END) AS viewed,
  COUNT(DISTINCT CASE WHEN e.event_name = 'add_to_cart' THEN u.user_id END) AS carted,
  COUNT(DISTINCT CASE WHEN e.event_name = 'checkout' THEN u.user_id END) AS checked_out,
  COUNT(DISTINCT CASE WHEN e.event_name = 'purchase' THEN u.user_id END) AS purchased,
  ROUND(100.0 * COUNT(DISTINCT CASE WHEN e.event_name = 'purchase' THEN u.user_id END)
      / COUNT(DISTINCT CASE WHEN e.event_name = 'view_item' THEN u.user_id END), 1) AS view_to_buy_pct
FROM app_users u
LEFT JOIN app_events e ON e.user_id = u.user_id
GROUP BY u.platform
ORDER BY u.platform
"""

def q5_pandas():
    reach = (app_events.drop_duplicates(["user_id", "event_name"])
             .merge(app_users[["user_id", "platform"]], on="user_id")
             .pivot_table(index="platform", columns="event_name", values="user_id",
                          aggfunc="count", fill_value=0))
    out = app_users.groupby("platform").size().rename("users").to_frame().join(reach).reset_index()
    out = out[["platform", "users", "view_item", "add_to_cart", "checkout", "purchase"]]
    out.columns = ["platform", "users", "viewed", "carted", "checked_out", "purchased"]
    out["view_to_buy_pct"] = (100 * out["purchased"] / out["viewed"]).round(1)
    return out
```

**Why:** `drop_duplicates` on (user, event) is the pandas form of COUNT(DISTINCT ...): after it, counting rows counts users. The users column comes from `app_users`, not from events, so users without events stay in the denominator (the LEFT JOIN in SQL). Web converts viewers to buyers far less (21.2% versus about 32%). Note that this unordered table can count a purchaser who never viewed; the ordered version (day 21) cannot.

**Complexity:** One join and one aggregation (SQL); one dedup and one pivot (pandas).

**Common mistakes:** `aggfunc='count'` without dedup (counts events, not users). `pivot` instead of `pivot_table` (fails on duplicate index pairs). Forgetting `fill_value=0`, which gives NaN for missing steps.

**Learn more:** [sql-pandas-equivalents](https://mahsa-agz.github.io/ds-handbook/#sql-pandas-equivalents), [sql-pivot](https://mahsa-agz.github.io/ds-handbook/#sql-pivot), [sql-funnels](https://mahsa-agz.github.io/ds-handbook/#sql-funnels)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_29/3_stats.ipynb)